# ✈️ Aria — WanderWise Travel | Agent-Powered Business Assistant

**Course:** EECE 503P/798S: Agentic Systems  
**Assignment C3:** Build an Agent-Powered Business Assistant  
**Supported LLM Providers:** Google Gemini (Free tier), OpenRouter (Free/budget models), OpenAI, AWS Bedrock  

## Project Overview
This notebook implements **Aria**, an agentic virtual travel concierge representing **WanderWise Travel** (a fictional boutique travel studio).  
Key capabilities demonstrated:
1. **Clearly Separated Identity**: Distinct **Persona** (tone & style) and **Instructions** (operational guardrails).
2. **Three-Way Intent Router**: Classifies queries into `general_question`, `lead_collection`, or `feedback_unanswered`.
3. **Documented MCP-Style Tools**: `record_customer_interest` and `record_feedback` with strict `inputSchema` and `outputSchema` definitions.
4. **Multi-Provider LLM Integration**: Runs with **Google Gemini**, **OpenRouter**, **OpenAI**, or **AWS Bedrock**.
5. **Interactive Gradio Interface**: Live web chat with an embedded **Live Log & Activity Inspector**.
6. **Reflection Pass**: Self-critique step that checks draft replies and logs original vs. revised answers.

> **Classroom Simulation Notice:** The agency, its packages and pricing ($1,850 Bali, $1,650 Rome, etc.) are fictional. The listed team members are the real student and course staff behind this EECE 503P/798S project (contact: contact@jr-web-lb.com). Aria does not book real flights or hotels.

### Environment Setup (local Jupyter or Google Colab)
Run the next cell first.

**Locally:** it does nothing extra. Keys are read from `.env` in the project folder.

**In Google Colab:** the notebook needs the rest of this repository (`agent_core.py`, `me/`), not just this file. Either:
1. keep `REPO_URL` below (it clones this repository), **or**
2. upload the whole `WanderWise-C3-Business-Agent` folder to Google Drive at `MyDrive/WanderWise-C3-Business-Agent`; the cell will ask to mount Drive.

Then add your API key in Colab's **Secrets** panel (the key icon in the left sidebar), for example `OPENROUTER_API_KEY`, and switch on "Notebook access". Optionally add `LLM_PROVIDER` too. Key values are never printed.

In [1]:
# --- Setup: works locally (Jupyter) and in Google Colab ---
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Jadjnm/WanderWise-C3-Business-Agent.git"  # Colab only; leave as is to clone this repository
DRIVE_DIR = Path("/content/drive/MyDrive/WanderWise-C3-Business-Agent")
SECRET_NAMES = ["LLM_PROVIDER", "OPENROUTER_API_KEY", "OPENROUTER_MODEL", "GEMINI_API_KEY", "GEMINI_MODEL",
                "OPENAI_API_KEY", "AWS_BEARER_TOKEN_BEDROCK", "AWS_REGION", "BEDROCK_MODEL_ID", "ENABLE_REFLECTION"]

if IN_COLAB:
    clone_dir = Path("/content/WanderWise-C3-Business-Agent")
    if REPO_URL and not clone_dir.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(clone_dir)], check=True)
    candidates = [clone_dir, DRIVE_DIR]
    if not any((c / "agent_core.py").exists() for c in candidates):
        from google.colab import drive
        drive.mount("/content/drive")
    project = next((c for c in candidates if (c / "agent_core.py").exists()), None)
    if project is None:
        raise FileNotFoundError(
            f"Project files not found. Set REPO_URL above, or upload the WanderWise_Agent_Project folder to {DRIVE_DIR}")
    os.chdir(project)

    # Install the project's requirements, except `notebook` (Colab ships its own Jupyter)
    reqs = [line.strip() for line in (project / "requirements.txt").read_text().splitlines()
            if line.strip() and not line.startswith("#") and not line.startswith("notebook")]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *reqs], check=True)

    # API keys from Colab Secrets (there is no .env file in Colab)
    from google.colab import userdata
    loaded = []
    for name in SECRET_NAMES:
        try:
            value = userdata.get(name)
        except Exception:  # secret not defined, or notebook access not granted
            value = None
        if value:
            os.environ[name] = value
            loaded.append(name)
    print("Secrets loaded:", ", ".join(loaded) or "none (add one in the Secrets panel)")

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
print(f"Project folder: {Path.cwd().name} | Colab: {IN_COLAB} | Python {sys.version.split()[0]}")

Project folder: WanderWise_Agent_Project | Colab: False | Python 3.14.3


## 1. Imports and Configuration
Load configuration from `.env`. The project automatically detects which provider key is active (`GEMINI_API_KEY`, `OPENROUTER_API_KEY`, `OPENAI_API_KEY`, or Bedrock credentials).

In [2]:
from agent_core import (
    BASE_DIR,
    LOG_DIR,
    get_active_provider,
    get_active_model,
    PERSONA,
    INSTRUCTIONS,
    load_business_knowledge,
    build_system_prompt,
    MCP_TOOL_DEFINITIONS,
    TOOL_SCHEMAS,
    record_customer_interest,
    record_feedback,
    route_message,
    route_message_with_metadata,
    run_agent,
    reflect_on_answer,
    build_demo,
)

print(f"Project folder: {BASE_DIR.name}")
print(f"Active LLM Provider: {get_active_provider().upper()}")
print(f"Active Model: {get_active_model()}")

Project folder: WanderWise_Agent_Project
Active LLM Provider: OPENAI
Active Model: gpt-4o-mini


## 2. PERSONA (Tone, Voice, and Style)
Defines who the agent is **separately** from operational instructions.

In [3]:
print("=== PERSONA ===")
print(PERSONA)

=== PERSONA ===
You are Aria, the friendly, enthusiastic, and thoughtful virtual travel concierge
for WanderWise Travel, a fictional boutique travel-planning business.
You sound like a knowledgeable, well-traveled friend: welcoming, curious,
practical, and never pushy. Your writing is warm and concise. A sparing travel
emoji is fine, but clarity matters more than cheerleading.


## 3. INSTRUCTIONS (Operational Guardrails)
Operational rules ensure Aria adheres to approved knowledge, collects customer inquiries without claiming live bookings, and calls logging tools when appropriate.

In [4]:
print("=== INSTRUCTIONS ===")
print(INSTRUCTIONS)

=== INSTRUCTIONS ===
1. Stay in character as Aria from WanderWise Travel.
2. Use ONLY the supplied business summary and business PDF extracts for factual
   claims about WanderWise: team, prices, packages, services, inclusions,
   policies, and contact methods. Never make up deals, dates, pricing, visas,
   availability, or reservation status.
3. For general inquiries, answer clearly using approved facts. When appropriate,
   ask about budget, interests, travel dates, and number of travelers, one or two at a time.
4. When someone wants a quote, consultation, or to book a trip, collect their name
   and email voluntarily, along with a brief description of their trip. Tell them this
   is an inquiry consultation, not a confirmed live booking. Once required details
   are given, call record_customer_interest(email, name, message). NEVER invent an email
   or name; politely ask if any required field is missing.
5. If the user submits a complaint, feedback, or asks an UNANSWERED business-sp

## 4. Business Knowledge (TXT + PDF)
Loads knowledge from `me/business_summary.txt` and `me/about_business.pdf`. Both source files are combined into the agent's system prompt.

In [5]:
knowledge = load_business_knowledge()
print(f"Total knowledge extracted: {len(knowledge)} characters.")
print("\n--- First 500 characters of Business Summary ---")
print(knowledge[:500] + "...")

Total knowledge extracted: 13021 characters.

--- First 500 characters of Business Summary ---
BUSINESS SUMMARY (primary reference):
WANDERWISE TRAVEL - OFFICIAL BUSINESS SUMMARY
CLASSROOM SIMULATION: This is a fictional agency. Packages, rates
and prices are invented for the EECE 503P/798S agent assignment; the team
members are the real student and course staff behind the project.
This assistant never makes real bookings or checks live availability.

BUSINESS IDENTITY
WanderWise Travel is a virtual-first boutique travel-planning studio with the
slogan "Your dream destination, thoughtfull...


## 5. Formal MCP-Style Tool Schemas
Documented according to the **Model Context Protocol (MCP)** specification: providing `name`, `description`, `inputSchema`, and `outputSchema`.

In [6]:
import json
print(json.dumps(MCP_TOOL_DEFINITIONS, indent=2))

[
  {
    "name": "record_customer_interest",
    "description": "Log a customer's voluntarily supplied contact details and vacation inquiry notes. Returns a structured JSON result.",
    "inputSchema": {
      "type": "object",
      "properties": {
        "email": {
          "type": "string",
          "format": "email",
          "description": "Customer's valid email address for follow-up."
        },
        "name": {
          "type": "string",
          "description": "Full name of the inquiring customer."
        },
        "message": {
          "type": "string",
          "description": "Summary of the trip details, destination, dates, budget, or travel style."
        }
      },
      "required": [
        "email",
        "name",
        "message"
      ],
      "additionalProperties": false
    },
    "outputSchema": {
      "type": "object",
      "properties": {
        "status": {
          "type": "string",
          "enum": [
            "success",
            "erro

## 6. Tool Implementations — Local JSONL Logging
These Python functions actually write records to `logs/leads.jsonl` and `logs/feedback.jsonl`.

In [7]:
# Test tool execution directly
lead_result = record_customer_interest(
    email="test_traveler@example.com",
    name="Alex Taylor",
    message="Looking for a 7-day honeymoon in Bali with rice terrace tours."
)
print("record_customer_interest result:", lead_result)

feedback_result = record_feedback(
    question="What is your policy regarding cancellation due to weather?"
)
print("record_feedback result:", feedback_result)

[TOOL] record_customer_interest: lead for 'Alex Taylor' saved to logs/leads.jsonl
record_customer_interest result: {'status': 'success', 'message': 'Travel inquiry saved for the demo team; no trip is booked.'}
[TOOL] record_feedback: entry saved to logs/feedback.jsonl
record_feedback result: {'status': 'success', 'message': 'Question/feedback recorded; no answer was invented.'}


## 7. Intent Router
Classifies every incoming message into exactly one of three paths:  
1. `general_question`: package details, pricing, team background.  
2. `lead_collection`: quote requests, consultation bookings, customer contact capture.  
3. `feedback_unanswered`: complaints, feedback, or unanswerable queries (cancellation, refunds, visa issuance, unannounced tours).

In [8]:
test_queries = [
    ("What is included in the Bali Bliss package and what does it cost?", "general_question"),
    ("I want to book a consultation for a Bali honeymoon in June.", "lead_collection"),
    ("What is your cancellation and refund policy?", "feedback_unanswered"),
    ("Will WanderWise launch Japan group tours next year?", "feedback_unanswered"),
    ("I have feedback: please offer more budget-friendly packages.", "feedback_unanswered"),
]

print("=== Router Evaluation ===\n")
for query, expected in test_queries:
    meta = route_message_with_metadata(query)
    matched = (meta["intent"] == expected)
    status = "PASS ✅" if matched else "FAIL ❌"
    print(f"{status} Query: \"{query}\"")
    print(f"       Route: {meta['intent']} (Rule: {meta['rule']})\n")
    assert matched, f"Expected {expected}, got {meta['intent']}"

=== Router Evaluation ===

PASS ✅ Query: "What is included in the Bali Bliss package and what does it cost?"
       Route: general_question (Rule: general_knowledge_query)

PASS ✅ Query: "I want to book a consultation for a Bali honeymoon in June."
       Route: lead_collection (Rule: lead_intent_detected)

PASS ✅ Query: "What is your cancellation and refund policy?"
       Route: feedback_unanswered (Rule: unsupported_cancellation_policy)

PASS ✅ Query: "Will WanderWise launch Japan group tours next year?"
       Route: feedback_unanswered (Rule: future_unannounced_offering)

PASS ✅ Query: "I have feedback: please offer more budget-friendly packages."
       Route: feedback_unanswered (Rule: explicit_feedback)



## 8. Reflection Step Demonstration
Before delivering a final reply, the agent can perform a self-correction pass checking its own draft against the instructions and knowledge base.

In [9]:
# Simulated draft with ungrounded claim vs reflected correction
imperfect_draft = (
    "We would love to book your flight tickets to Bali right now! "
    "Our cancellation policy allows full refunds up to 24 hours prior."
)

print("Original Draft (contains ungrounded flight booking & cancellation claims):")
print(f"> {imperfect_draft}\n")

corrected_reply = reflect_on_answer(
    draft=imperfect_draft,
    intent="feedback_unanswered",
    provider=get_active_provider()
)

print("Reflected / Self-Corrected Reply:")
print(f"> {corrected_reply}\n")
print("Both replies are recorded in logs/reflections.jsonl for evaluation.")

Original Draft (contains ungrounded flight booking & cancellation claims):
> We would love to book your flight tickets to Bali right now! Our cancellation policy allows full refunds up to 24 hours prior.



[BONUS] Reflection logged (original draft and revised reply saved to logs/reflections.jsonl).
Reflected / Self-Corrected Reply:
> I’m glad to hear you’re interested in Bali! However, I must clarify that WanderWise Travel cannot book flight tickets or confirm availability at this time. Additionally, we currently do not have a cancellation policy outlined for our packages. If you'd like assistance planning your trip, please share your name, email, and a brief description of your travel preferences, and I can log your inquiry for follow-up.

Both replies are recorded in logs/reflections.jsonl for evaluation.


## 9. Launch Aria's Chatbot & Live Inspector (Gradio)
Run the cell below to launch the interactive Gradio demo.  
Features:
- **Tab 1: Travel Concierge Chat** with quick example prompts.
- **Tab 2: Live Logs Inspector** to see real-time updates to `leads.jsonl`, `feedback.jsonl`, and `reflections.jsonl`.

In [10]:
# Launch Gradio demo
demo = build_demo()
demo.launch(share=False)

* Running on local URL:  http://127.0.0.1:7862


* To create a public link, set `share=True` in `launch()`.


## 10. Submission Video Checklist

When recording your 2–3 minute submission video, follow this sequence:
1. **Identity (20s):** Show `me/about_business.pdf` and `me/business_summary.txt`.
2. **Persona & Instructions (20s):** Highlight the distinct `PERSONA` and `INSTRUCTIONS` blocks in this notebook.
3. **General Question (30s):** Ask about Bali Bliss pricing ($1,850/person) and inclusions (`general_question` route).
4. **Lead Collection (40s):** Ask to book a consultation, provide contact details, show `record_customer_interest` executing, and display the new lead in the **Live Logs Inspector** tab.
5. **Feedback / Unanswered Question (30s):** Ask about cancellation policy or unannounced Japan group tours; show `record_feedback` executing and logging.
6. **Reflection Pass (20s):** Show the logged self-correction in `logs/reflections.jsonl`.
7. **Hugging Face Spaces (20s):** Show your deployed Gradio app running on Hugging Face Spaces.